# Klasyczna homografia boiska — PEŁNY AUTOMAT, krok po kroku

Estymacja **H (piksel → metry)** dla pojedynczej klatki, w pełni automatycznie (bez klikania).
Część klatek świadomie zwróci `None` — to oczekiwane, gdy widać za mało/niejednoznaczne cechy.

Potok:
1. Segmentacja pola → 2. Maska linii → 3. Odcinki → 4. Scalanie → 5. Grupy + punkty zbiegu
5b. Wykrycie koła środkowego (elipsa)
6. **Automat**: metoda z kołem (gdy widoczne) lub VP‑RANSAC; wybór po F1; `None` poniżej progu
7. Refinement + projekcja na boisko

> Realia: na pojedynczym, zoom‑owanym ujęciu pola karnego (siatka bramki, mało linii) lub przy kole
> zasłoniętym przez zawodników automat często zwróci `None`. W potoku wideo pełne pokrycie uzyskuje się
> przez **propagację H** z klatek o czystych cechach na sąsiednie (śledzenie cech) — bez udziału człowieka.
> Na końcu jest opcjonalny **fallback ręczny** (korespondencja) na wypadek, gdy automat zawiedzie.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import cv2
import numpy as np
import matplotlib.pyplot as plt

try:
    from src.calibration.classical_homography_v2 import ClassicalHomographyV2
except ModuleNotFoundError:
    from classical_homography_v2 import ClassicalHomographyV2   # fallback: plik obok notebooka

%matplotlib inline
plt.rcParams['figure.figsize'] = (16, 9)
plt.rcParams['figure.dpi'] = 110


def show(img_bgr=None, mask=None, title='', ax=None, axis=True):
    own = ax is None
    if own:
        _, ax = plt.subplots(figsize=(14, 8))
    if mask is not None:
        ax.imshow(mask, cmap='gray')
    else:
        ax.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    if not axis:
        ax.axis('off')
    if own:
        plt.show()


def draw_groups(image, group_a, group_b, inter=None):
    vis = image.copy()
    def _ext(m):
        d = m['direction']; p1 = m['p1'] - 4000 * d; p2 = m['p2'] + 4000 * d
        return (int(p1[0]), int(p1[1])), (int(p2[0]), int(p2[1]))
    for m in group_a:
        a, b = _ext(m); cv2.line(vis, a, b, (0, 0, 255), 2)
    for m in group_b:
        a, b = _ext(m); cv2.line(vis, a, b, (255, 128, 0), 2)
    if inter is not None:
        for i, p in enumerate(inter):
            cv2.circle(vis, (int(p[0]), int(p[1])), 7, (0, 255, 255), -1)
    return vis

## Krok 0 — wybór i wczytanie klatki

In [ ]:
# Zmien FRAME_NUM, by przetestowac rozne klatki. Domyslnie szuka w repo, potem obok notebooka.
SEQUENCE = "SNMOT-123"
FRAME_NUM = 299

frame_path = (PROJECT_ROOT / "data" / "tracking_dataset" / "tracking" / "test"
              / SEQUENCE / "img1" / f"{FRAME_NUM:06d}.jpg")
if not frame_path.exists():
    frame_path = Path.cwd() / f"{FRAME_NUM:06d}.jpg"

image = cv2.imread(str(frame_path))
assert image is not None, f"Nie udalo sie wczytac: {frame_path}"
h, w = image.shape[:2]
print(f"Obraz: {w}x{h}  ({frame_path})")

cal = ClassicalHomographyV2(image_width=w, image_height=h,
                            ransac_iterations=2000, min_score=0.30, verbose=True)
show(image, title=f"Klatka: {frame_path.name}", axis=False)

## Krok 1 — segmentacja pola gry

In [ ]:
field_mask, hull = cal.segment_field(image)
vis_hull = image.copy()
if hull is not None:
    cv2.polylines(vis_hull, [hull], True, (0, 0, 255), 3)
fig, axes = plt.subplots(1, 2, figsize=(20, 7))
show(vis_hull, title="Convex hull pola", ax=axes[0], axis=False)
show(mask=field_mask, title=f"Maska pola ({np.mean(field_mask > 0) * 100:.1f}% kadru)", ax=axes[1])
axes[1].axis('off'); plt.tight_layout(); plt.show()

## Krok 2 — maska białych linii

In [ ]:
line_mask = cal.extract_line_mask(image, field_mask)
print(f"Bialych pikseli linii: {int((line_mask > 0).sum())}")
show(mask=line_mask, title="Maska bialych linii")

## Krok 3 — detekcja odcinków (HoughLinesP)

In [ ]:
segments = cal.detect_segments(line_mask)
print(f"Odcinki: {len(segments)}")
vis = image.copy()
for x1, y1, x2, y2 in segments:
    cv2.line(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
show(vis, title=f"Odcinki: {len(segments)}", axis=False)

## Krok 4 — scalanie kolinearnych odcinków

In [ ]:
merged = cal.merge_collinear(segments)
print(f"Linie po scaleniu: {len(merged)}")
vis = image.copy()
for m in merged:
    p1, p2 = m['p1'].astype(int), m['p2'].astype(int)
    cv2.line(vis, tuple(p1), tuple(p2), (0, 255, 0), 3)
show(vis, title=f"Scalone linie: {len(merged)}", axis=False)
for i, m in enumerate(merged):
    print(f"  [{i}] kat={m['angle']:6.1f}  dlugosc={m['total_length']:6.0f}")

## Krok 5 — grupowanie w 2 kierunki + punkty zbiegu

In [ ]:
group_a, group_b = cal.group_two_directions(merged)
vp_a = cal.compute_vanishing_point(group_a)
vp_b = cal.compute_vanishing_point(group_b)
inter = cal.detect_intersections(group_a, group_b)
print(f"Grupa A: {len(group_a)} | Grupa B: {len(group_b)} | przeciec: {len(inter)}")
show(draw_groups(image, group_a, group_b, inter),
     title="Dwa kierunki linii (A=czerwony, B=niebieski) + przeciecia (zolte)", axis=False)

## Krok 5b — wykrycie koła środkowego (elipsa)
Usuwamy piksele leżące na prostych liniach (zostaje łuk koła) i dopasowujemy elipsę (RANSAC, z walidacją
rozmiaru/pozycji). Koło daje skalę metryczną i dodatkowe korespondencje (trik z biegunem punktu zbiegu).

In [ ]:
C, ell = cal.detect_circle(line_mask, merged)
if ell is not None:
    xc, yc, a, b, th = ell
    vis = image.copy()
    cv2.ellipse(vis, (int(xc), int(yc)), (int(a), int(b)), np.degrees(th), 0, 360, (0, 255, 255), 3)
    show(vis, title=f"Wykryta elipsa: c=({xc:.0f},{yc:.0f}) axes=({a:.0f},{b:.0f})", axis=False)
    print("Uwaga: gdy kolo jest mocno zaslonene przez zawodnikow, dopasowanie bywa bledne -> automat to odrzuci.")
else:
    print("Nie wykryto kola (brak/za krotki luk — typowe dla ujec pola karnego). Automat uzyje VP-RANSAC.")

## Krok 6 — PEŁNY AUTOMAT (bez klikania)
`estimate_homography` próbuje metody z kołem (gdy wykryte) i VP‑RANSAC, wybiera lepszy wynik po F1.
Jeśli najlepsze F1 < `min_score`, zwracamy `None` — świadoma „porażka", zamiast błędnej homografii.

In [ ]:
H, score = cal.estimate_homography(line_mask, group_a, group_b)
print(f"Najlepsze F1 = {score:.3f}   (prog min_score = {cal.min_score})")

if H is not None and score >= cal.min_score:
    if cal.refine_steps > 0:
        H, score = cal.refine_homography(H, line_mask)
    show(cal.draw_overlay(image, H, thickness=2),
         title=f"PELNY AUTOMAT — model boiska nalozony (F1={score:.3f})", axis=False)
    print("Macierz H (pixel -> metry):")
    print(np.array2string(H, precision=4, suppress_small=True))
else:
    H = None
    print("AUTOMAT: brak pewnego H na tej klatce -> None.")
    print("Przyczyny: zbyt malo widocznych linii / siatka bramki w masce / zaslonene kolo.")
    print("To akceptowalne: w potoku wideo te klatki obsluguje sie propagacja H z sasiednich.")

## Krok 7 — projekcja na boisko (gdy H znalezione)

In [ ]:
if H is not None:
    fig, axes = plt.subplots(1, 2, figsize=(22, 7))
    show(cal.draw_overlay(image, H, thickness=2), title="Nakladka modelu", ax=axes[0], axis=False)
    ax = axes[1]
    for poly in cal.pitch_lines:
        p = np.array(poly); ax.plot(p[:, 0], p[:, 1], 'k-', lw=0.8)
    for px, py in [(w*0.5, h*0.72), (w*0.3, h*0.82), (w*0.7, h*0.62)]:
        xy = cal.project_point_to_pitch((px, py), H)
        if xy:
            ax.plot(xy[0], xy[1], 'ro', ms=9)
    ax.set_aspect('equal'); ax.invert_yaxis()
    ax.set_xlim(-56, 56); ax.set_ylim(36, -36)
    ax.set_title("Projekcja punktow stop -> boisko [m]")
    plt.tight_layout(); plt.show()
else:
    print("Brak H — nic do rzutowania. (Patrz dodatek: opcjonalny fallback reczny.)")

## Dodatek (opcjonalny) — ręczny fallback, gdy automat zwróci `None`
Nie jest częścią automatu. Pozwala dograć trudną klatkę kilkoma punktami (klik lub ręcznie),
przez `findHomography`. Uzupełnij `CLICKS` (≥4) nazwami z `cal.named_keypoints`.

In [ ]:
# Opcjonalny klikacz (Jupyter z ipympl / Colab):
LANDMARKS = ["C_top", "C_center", "C_bot",          # linia srodkowa: gora / srodek / dol
             "L_box_front_top", "L_box_front_bot"]   # lub rogi pola karnego itp.
clicked_px = {}
def make_clicker():
    fig, ax = plt.subplots(figsize=(16, 9)); ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    st = {"i": 0}; ax.set_title(f"Kliknij: {LANDMARKS[0]}")
    def onclick(ev):
        if ev.xdata is None or st["i"] >= len(LANDMARKS): return
        nm = LANDMARKS[st["i"]]; clicked_px[nm] = (float(ev.xdata), float(ev.ydata))
        ax.plot(ev.xdata, ev.ydata, 'yo'); st["i"] += 1
        ax.set_title(f"Kliknij: {LANDMARKS[st['i']]}" if st["i"] < len(LANDMARKS) else "Gotowe")
        fig.canvas.draw_idle()
    fig.canvas.mpl_connect("button_press_event", onclick); return fig
# %matplotlib widget
# make_clicker()

CLICKS = dict(clicked_px)
CLICKS.update({
    # "C_top": (..,..), "C_bot": (..,..), ...  # uzupelnij recznie odczytujac z osi obrazu
})
if len(CLICKS) >= 4:
    names = list(CLICKS)
    ip = np.array([CLICKS[k] for k in names], float)
    mp = np.array([cal.named_keypoints[k] for k in names], float)
    Hm, sm = cal.estimate_homography_from_correspondences(ip, mp, line_mask=line_mask)
    print(f"H reczne: F1={sm:.3f}")
    if Hm is not None:
        show(cal.draw_overlay(image, Hm, thickness=2), title=f"FALLBACK reczny (F1={sm:.3f})", axis=False)
else:
    print(f"Fallback nieaktywny ({len(CLICKS)} punktow). Uzupelnij CLICKS do >=4, jesli automat dal None.")